In [1]:
import stim

In [2]:
import torch
import torch.nn.functional as F


In [3]:
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.utils import add_self_loops, degree
from torch_geometric.utils import softmax  # For attention normalization



In [4]:
import matplotlib.pyplot as plt


In [5]:
import numpy as np
from tqdm.auto import trange
import networkx as nx
import math

c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# ============================================================================
# PART 1: SURFACE CODE CIRCUIT
# ============================================================================

def surface_code_circuit(p, d):
    """Generate surface code circuit with given error rate and distance"""
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=d,
        distance=d,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p
    )

In [8]:
class SyndromeGraph:
    """
    Pre-built graph structure for a surface code.
    NOW TRACKS EDGE TYPES: 0=spatial, 1=temporal
    AND INCLUDES RICH NODE FEATURES: stabilizer_type, position, time
    """

    def __init__(self, circuit, device):
        self.device = device
        self.num_nodes = circuit.num_detectors

        # Build the fixed graph skeleton ONCE (now returns edge types too)
        self.edge_index, self.edge_types = self._build_edges(circuit)
        self.node_features = self._build_node_features(circuit)  # Static features

        self.edge_index = self.edge_index.to(device)
        self.edge_types = self.edge_types.to(device)
        self.node_features = self.node_features.to(device)

        self.num_edges = self.edge_index.shape[1]
        self.num_node_features = self.node_features.shape[1] + 1  # +1 for syndrome

        print(f"✓ Graph skeleton built: {self.num_nodes} detectors, {self.num_edges} connections")
        print(f"  Spatial edges: {(self.edge_types == 0).sum().item()}")
        print(f"  Temporal edges: {(self.edge_types == 1).sum().item()}")
        print(f"  Node features: {self.num_node_features} (syndrome + {self.node_features.shape[1]} static)")
        print(f"  Average neighbors per detector: {self.num_edges / self.num_nodes:.1f}")

    def _build_node_features(self, circuit):
        """
        Build static node features from detector coordinates.

        Features per node:
        - stabilizer_type: 0=X, 1=Z (based on checkerboard pattern)
        - dist_north: normalized distance from north boundary
        - dist_west: normalized distance from west boundary
        - time_round: normalized time round
        """
        detector_coords = circuit.get_detector_coordinates()

        # Get coordinate ranges for normalization
        all_coords = list(detector_coords.values())
        xs = [c[0] for c in all_coords]
        ys = [c[1] for c in all_coords]
        ts = [c[2] for c in all_coords]

        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        t_max = max(ts)

        features = []
        for det_id in range(self.num_nodes):
            coord = detector_coords.get(det_id, [0, 0, 0])
            x, y, t = coord[0], coord[1], coord[2]

            # Stabilizer type: In rotated surface code, X and Z alternate
            # Based on checkerboard pattern of (x + y) / 2
            stabilizer_type = ((x + y) / 2) % 2  # 0 or 1

            # Normalized distances from boundaries (0 to 1)
            dist_north = (y - y_min) / max(1, y_max - y_min)
            dist_west = (x - x_min) / max(1, x_max - x_min)

            # Normalized time round (0 to 1)
            time_norm = t / max(1, t_max)

            features.append([stabilizer_type, dist_north, dist_west, time_norm])

        return torch.tensor(features, dtype=torch.float)

    def _build_edges(self, circuit):
        """
        Build edge connections AND edge types.
        Returns: (edge_index, edge_types)
        """
        detector_coords = circuit.get_detector_coordinates()
        edges = []
        edge_types = []  # 0 = spatial, 1 = temporal

        # Build lookup: (x, y, t) -> detector_id for temporal connections
        coord_to_id = {}
        for det_id, coord in detector_coords.items():
            key = (coord[0], coord[1], coord[2])
            coord_to_id[key] = det_id

        for i in range(self.num_nodes):
            coord_i = detector_coords.get(i, [0, 0, 0])
            xi, yi, ti = coord_i[0], coord_i[1], coord_i[2]

            for j in range(i + 1, self.num_nodes):
                coord_j = detector_coords.get(j, [0, 0, 0])
                xj, yj, tj = coord_j[0], coord_j[1], coord_j[2]

                # Spatial connection: same time round, adjacent in space
                if ti == tj:
                    dx = abs(xi - xj)
                    dy = abs(yi - yj)

                    if (dx == 2 and dy == 0) or (dx == 0 and dy == 2):
                        edges.append([i, j])
                        edges.append([j, i])
                        edge_types.extend([0, 0])  # Both directions are spatial

            # Temporal connection: same spatial position, next time round
            next_key = (xi, yi, ti + 1)
            if next_key in coord_to_id:
                j = coord_to_id[next_key]
                edges.append([i, j])
                edge_types.append(1)  # Temporal

        if len(edges) == 0:
            print("  Warning: No edges found, using fallback chain")
            for i in range(self.num_nodes - 1):
                edges.append([i, i + 1])
                edges.append([i + 1, i])
                edge_types.extend([0, 0])

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_types = torch.tensor(edge_types, dtype=torch.long)
        return edge_index, edge_types

    def create_batch(self, syndrome_values):
        """
        Create batched graph data from syndrome measurements.
        NOW INCLUDES RICH NODE FEATURES: [syndrome, stabilizer_type, dist_north, dist_west, time]
        """
        batch_size = syndrome_values.shape[0]

        # Syndrome values: [batch, nodes] -> [batch*nodes, 1]
        syndrome_feat = syndrome_values[:, :self.num_nodes].reshape(-1, 1).float()

        # Static features: repeat for each sample in batch [batch*nodes, 4]
        static_feat = self.node_features.repeat(batch_size, 1)

        # Concatenate all features: [batch*nodes, 5]
        x = torch.cat([syndrome_feat, static_feat], dim=1)

        # Replicate edges and edge types for each graph in batch
        edge_indices = []
        edge_types_list = []
        for i in range(batch_size):
            offset = i * self.num_nodes
            edge_indices.append(self.edge_index + offset)
            edge_types_list.append(self.edge_types)

        edge_index = torch.cat(edge_indices, dim=1)
        edge_type = torch.cat(edge_types_list, dim=0)

        # Batch assignment
        batch = torch.arange(batch_size, device=self.device).repeat_interleave(self.num_nodes)

        return Data(x=x, edge_index=edge_index, edge_type=edge_type, batch=batch)


In [9]:
class GATConvWithEdgeType(MessagePassing):
    """
    Graph Attention Layer with edge type embeddings.

    Attention score for edge i->j with type t:
        α_ij = softmax(LeakyReLU(a^T [W*h_i || W*h_j || edge_emb_t]))
    """

    def __init__(self, in_channels, out_channels, heads=4, concat=True,
                 dropout=0.0, num_edge_types=2):
        super().__init__(aggr='add', node_dim=0)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.dropout = dropout
        self.num_edge_types = num_edge_types

        # Linear transformation for node features
        self.lin = torch.nn.Linear(in_channels, heads * out_channels, bias=False)

        # Edge type embeddings
        self.edge_type_emb = torch.nn.Embedding(num_edge_types, heads * out_channels)

        # Attention mechanism: a^T [source || target || edge_type]
        self.att = torch.nn.Parameter(torch.Tensor(1, heads, 3 * out_channels))

        # Bias
        if concat:
            self.bias = torch.nn.Parameter(torch.Tensor(heads * out_channels))
        else:
            self.bias = torch.nn.Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.lin.weight)
        torch.nn.init.xavier_uniform_(self.att)
        torch.nn.init.xavier_uniform_(self.edge_type_emb.weight)
        torch.nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_type):
        """
        Args:
            x: Node features [num_nodes, in_channels]
            edge_index: Edge indices [2, num_edges]
            edge_type: Edge types [num_edges] (0=spatial, 1=temporal)
        """
        # Transform node features
        x = self.lin(x).view(-1, self.heads, self.out_channels)

        # Get edge type embeddings
        edge_emb = self.edge_type_emb(edge_type).view(-1, self.heads, self.out_channels)

        # Start propagating messages
        out = self.propagate(edge_index, x=x, edge_emb=edge_emb)

        # Concatenate or average heads
        if self.concat:
            out = out.view(-1, self.heads * self.out_channels)
        else:
            out = out.mean(dim=1)

        return out + self.bias

    def message(self, x_i, x_j, edge_emb, index, ptr, size_i):
        """
        Compute attention coefficients and messages.

        x_i: Target node features [num_edges, heads, out_channels]
        x_j: Source node features [num_edges, heads, out_channels]
        edge_emb: Edge type embeddings [num_edges, heads, out_channels]
        """
        # Concatenate source, target, and edge type: [num_edges, heads, 3*out_channels]
        alpha_input = torch.cat([x_i, x_j, edge_emb], dim=-1)

        # Compute attention coefficients: [num_edges, heads]
        alpha = (alpha_input * self.att).sum(dim=-1)
        alpha = F.leaky_relu(alpha, negative_slope=0.2)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        # Apply attention to source features
        return x_j * alpha.unsqueeze(-1)


In [10]:

class SyndromeDecoder(torch.nn.Module):
    """
    GNN with attention that takes syndrome graphs and predicts logical error.
    Now accepts 5 input features: [syndrome, stabilizer_type, dist_north, dist_west, time]
    """

    def __init__(self, hidden_dim=128, num_layers=4, num_heads=4, num_input_features=5):
        super().__init__()

        # Attention layers
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        # First layer: num_input_features -> hidden
        self.layers.append(GATConvWithEdgeType(
            num_input_features, hidden_dim // num_heads,
            heads=num_heads,
            concat=True,
            num_edge_types=2
        ))
        self.norms.append(torch.nn.BatchNorm1d(hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 1):
            self.layers.append(GATConvWithEdgeType(
                hidden_dim, hidden_dim // num_heads,
                heads=num_heads,
                concat=True,
                num_edge_types=2
            ))
            self.norms.append(torch.nn.BatchNorm1d(hidden_dim))

        # Output: aggregate graph -> predict error
        self.output = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 64),
            torch.nn.SiLU(),
            torch.nn.Linear(64, 1)
        )

    def forward(self, data):
        x, edge_index, edge_type, batch = data.x, data.edge_index, data.edge_type, data.batch

        # Attention-based message passing
        for layer, norm in zip(self.layers, self.norms):
            x = layer(x, edge_index, edge_type)
            x = norm(x)
            x = F.silu(x)

        # Pool: aggregate all nodes in each graph
        graph_features = global_mean_pool(x, batch)

        # Predict logical error
        return self.output(graph_features)

In [11]:
# ============================================================================
# PART 5: GNN TRAINER - Clean interface with progress tracking
# ============================================================================

class GNNTrainer:
    """Clean interface for training GNN decoder with clear progress tracking"""

    def __init__(self, p, d, device, hidden_dim=128, num_layers=4):
        self.p = p
        self.d = d
        self.device = device

        print(f"\n{'='*60}")
        print(f"Initializing GNN Trainer for d={d}, p={p}")
        print(f"{'='*60}")

        # Build circuit (ONCE)
        print("\n[1/3] Building quantum circuit...")
        self.circuit = surface_code_circuit(p, d)

        # Build graph structure (ONCE)
        print("\n[2/3] Building graph skeleton...")
        self.graph = SyndromeGraph(self.circuit, device)

        # Build model
        print("\n[3/3] Building neural network...")
        self.model = SyndromeDecoder(hidden_dim=hidden_dim, num_layers=num_layers).to(device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=3e-4)
        # Note: loss_fn will be set dynamically in train_epoch to handle class imbalance

        # Count parameters
        num_params = sum(p.numel() for p in self.model.parameters())
        print(f"✓ Model built: {num_params:,} parameters")
        print(f"\n{'='*60}")
        print("Ready to train!")
        print(f"{'='*60}\n")

    def sample_data(self, num_samples):
        """Generate syndrome samples"""
        sampler = self.circuit.compile_detector_sampler()
        detections, flips = sampler.sample(num_samples, separate_observables=True)

        # Convert: 0/1 -> -1/+1
        # Use .copy() to ensure contiguous array, then torch.from_numpy() to avoid dtype inference issues
        syndrome_np = (detections.astype(np.int64) * 2 - 1).copy()
        labels_np = flips.astype(np.int64).flatten().copy()

        syndromes = torch.from_numpy(syndrome_np).to(self.device)
        labels = torch.from_numpy(labels_np).to(self.device)

        return syndromes, labels

    def train_epoch(self, syndromes, labels, batch_size=256):
        """One training epoch with progress bar"""
        self.model.train()
        num_samples = syndromes.shape[0]
        num_batches = math.ceil(num_samples / batch_size)

        # Compute class weights for imbalanced data
        # At low error rates, positive class (errors) is rare
        num_pos = labels.sum().item()
        num_neg = num_samples - num_pos
        if num_pos > 0 and num_neg > 0:
            pos_weight = torch.tensor([num_neg / num_pos], device=self.device)
        else:
            pos_weight = torch.tensor([1.0], device=self.device)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        total_loss = 0
        correct = 0
        samples_seen = 0

        with trange(num_batches, desc="Training", leave=False) as pbar:
            for batch_idx in pbar:
                start = batch_idx * batch_size
                end = min(start + batch_size, num_samples)  # Handle last batch

                batch_syn = syndromes[start:end]
                batch_lab = labels[start:end]
                batch_len = end - start  # Actual batch size

                # Create graph batch (structure pre-built, just plug in values)
                batch_data = self.graph.create_batch(batch_syn)

                # Forward (get logits, not probabilities)
                logits = self.model(batch_data)
                loss = loss_fn(logits, batch_lab.unsqueeze(1).float())

                # Backward
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                # Track metrics (convert logits to predictions)
                total_loss += loss.item()
                pred = torch.sigmoid(logits)
                batch_correct = ((pred > 0.5).flatten() == batch_lab).sum().item()
                correct += batch_correct
                samples_seen += batch_len

                # Update progress bar with correct running accuracy
                running_acc = correct / samples_seen
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{running_acc:.4f}'
                })

        avg_loss = total_loss / num_batches
        accuracy = correct / num_samples  # Use actual sample count
        return avg_loss, accuracy

    def evaluate(self, num_samples=10000, batch_size=500):
        """Evaluate on fresh data"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_samples)

        all_preds = []
        with torch.no_grad():
            for i in range(0, num_samples, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                logits = self.model(batch_data)
                pred = torch.sigmoid(logits)
                all_preds.append((pred > 0.5).flatten())

        all_preds = torch.cat(all_preds)
        accuracy = (all_preds == labels).float().mean().item()
        return accuracy

    def get_ler(self, num_shots=100000, batch_size=1000):
        """Compute logical error rate"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_shots)

        all_preds = []
        with torch.no_grad():
            for i in range(0, num_shots, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                logits = self.model(batch_data)
                pred = torch.sigmoid(logits)
                all_preds.append((pred > 0.5).flatten().cpu().numpy())

        all_preds = np.concatenate(all_preds)
        errors = (all_preds != labels.cpu().numpy())
        return errors.mean()

    def save(self, path):
        """Save model weights"""
        torch.save(self.model.state_dict(), path)
        print(f"✓ Model saved to {path}")

    def load(self, path):
        """Load model weights"""
        self.model.load_state_dict(torch.load(path, weights_only=True))
        print(f"✓ Model loaded from {path}")

In [12]:
# ============================================================================
# PART 6: MWPM BASELINE FOR COMPARISON
# ============================================================================

def get_mwpm_accuracy(p, d, num_shots=100000):
    """Compute MWPM accuracy for comparison"""
    import pymatching

    print(f"Computing MWPM baseline ({num_shots:,} shots)...")

    circuit = surface_code_circuit(p, d)
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    predictions = matcher.decode_batch(detection_events)

    num_errors = sum(
        1 for shot in range(num_shots)
        if not np.array_equal(observable_flips[shot], predictions[shot])
    )

    ler = num_errors / num_shots
    accuracy = 1 - ler
    print(f"✓ MWPM Accuracy: {accuracy:.6f} (LER: {ler:.6f})")

    return accuracy

In [13]:
# ============================================================================
# PART 7: PROGRESSIVE TRAINING - Train until beating MWPM
# ============================================================================

def train_until_beat_mwpm(p, d, device, max_train_size=10**8, chunk_size=10**7):
    """Train with progressively larger datasets until beating MWPM"""
    import gc

    print(f"\n{'#'*60}")
    print(f"# PROGRESSIVE TRAINING: d={d}, p={p}")
    print(f"{'#'*60}")

    # Get MWPM baseline first
    print("\n" + "="*60)
    print("STEP 1: Getting MWPM baseline")
    print("="*60)
    mwpm_accuracy = get_mwpm_accuracy(p, d)
    print(f"\n🎯 Target to beat: {mwpm_accuracy:.6f}")

    # Initialize trainer
    print("\n" + "="*60)
    print("STEP 2: Setting up GNN")
    print("="*60)
    trainer = GNNTrainer(p, d, device)

    # Progressive training
    print("\n" + "="*60)
    print("STEP 3: Progressive training")
    print("="*60)

    train_size = 100
    beat_mwpm = False

    while train_size <= max_train_size and not beat_mwpm:
        print(f"\n{'─'*60}")
        print(f"📊 Training with {train_size:,} samples")
        print(f"{'─'*60}")

        # Reset model for fresh start
        trainer.model = SyndromeDecoder(hidden_dim=128, num_layers=4).to(device)
        trainer.optimizer = torch.optim.Adam(trainer.model.parameters(), lr=3e-4)

        # Calculate chunks needed
        num_chunks = max(1, train_size // chunk_size)
        samples_per_chunk = min(train_size, chunk_size)

        # Train in chunks
        for chunk_idx in range(num_chunks):
            current_chunk_size = min(samples_per_chunk, train_size - chunk_idx * chunk_size)

            print(f"\n  Chunk {chunk_idx + 1}/{num_chunks}: {current_chunk_size:,} samples")
            print(f"  Generating data...")

            syndromes, labels = trainer.sample_data(current_chunk_size)

            # Multiple epochs per chunk for small datasets
            num_epochs = max(1, min(10, 10000 // (current_chunk_size // 256 + 1)))

            for epoch in range(num_epochs):
                loss, acc = trainer.train_epoch(syndromes, labels, batch_size=256)
                print(f"    Epoch {epoch+1}/{num_epochs}: Loss={loss:.4f}, Train Acc={acc:.4f}")

            # Cleanup
            del syndromes, labels
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Evaluate
        print(f"\n  Evaluating on fresh test data...")
        gnn_accuracy = trainer.evaluate(num_samples=10000)

        print(f"\n  📈 Results:")
        print(f"     GNN Accuracy:  {gnn_accuracy:.6f}")
        print(f"     MWPM Accuracy: {mwpm_accuracy:.6f}")
        print(f"     Difference:    {(gnn_accuracy - mwpm_accuracy)*100:+.3f}%")

        if gnn_accuracy > mwpm_accuracy:
            print(f"\n  🎉 SUCCESS! GNN beats MWPM at train_size = {train_size:,}")
            beat_mwpm = True
        else:
            print(f"\n  ❌ Not yet. Trying 10x more data...")
            train_size *= 10
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not beat_mwpm:
        print(f"\n❌ Did not beat MWPM within max train_size = {max_train_size:,}")
        return None, None

    return trainer, train_size

In [ ]:
# ============================================================================
# PART 8: RUN THE EXPERIMENT
# ============================================================================

import os
from datetime import datetime

def log_training_result(d, p, train_size, gnn_accuracy, mwpm_accuracy, training_time, status, notes):
    """Append training result to TRAINING_LOG.md

    gnn_accuracy=None indicates interrupted (will show as '—' in log)
    train_size can be int or string like "partial"
    status: "completed", "interrupted", "loaded"
    """
    log_file = "TRAINING_LOG.md"
    date_str = datetime.now().strftime("%Y-%m-%d %H:%M")

    # Format values
    if gnn_accuracy is not None:
        gnn_acc_str = f"{gnn_accuracy:.6f}"
        if mwpm_accuracy is not None:
            diff = (gnn_accuracy - mwpm_accuracy) * 100
            diff_str = f"{diff:+.2f}"
        else:
            diff_str = "—"
    else:
        gnn_acc_str = "—"
        diff_str = "—"

    mwpm_acc_str = f"{mwpm_accuracy:.6f}" if mwpm_accuracy is not None else "—"
    if isinstance(train_size, (int, float)):
        train_size_str = f"{train_size:,.0f}"
    else:
        train_size_str = str(train_size)

    # Format row
    row = f"| {date_str} | {d} | {p} | {train_size_str} | {gnn_acc_str} | {mwpm_acc_str} | {diff_str} | {training_time:.1f}m | {status} | {notes} |\n"

    # Read current file
    if os.path.exists(log_file):
        with open(log_file, 'r') as f:
            lines = f.readlines()
    else:
        lines = []

    # Find insertion point (after header)
    insert_idx = 2
    for i, line in enumerate(lines):
        if line.strip().startswith("| | | |"):
            insert_idx = i
            break

    lines.insert(insert_idx, row)

    # Write back
    with open(log_file, 'w') as f:
        f.writelines(lines)

    print(f"✓ Logged to TRAINING_LOG.md")


def get_descriptive_model_name(d, p, features=5, layers=4, hidden_dim=128):
    """Generate descriptive model filename"""
    return f"gnn_d{d}_p{p}_feat{features}_layer{layers}_hidden{hidden_dim}.pt"

from pathlib import Path

# Configuration
train_p = 0.005
max_train_size = 10**8
chunk_size = 10**7

results = {}
trained_models = {}
start_time = datetime.now()


# Create weights directory
weights_dir = str(Path(__file__).parent / "weights")
if not os.path.exists(weights_dir):
    os.makedirs(weights_dir)
    print(f"✓ Created {weights_dir}/ directory")

try:
    # Train for each distance
    for d in [3, 5, 7]:
        model_path = os.path.join(weights_dir, get_descriptive_model_name(d, train_p))

        print(f"\n{'█'*60}")
        print(f"█  DISTANCE {d}")
        print(f"{'█'*60}")

        # Check if model already exists
        if os.path.exists(model_path):
            print(f"\n✓ Found existing model: {model_path}")
            trainer = GNNTrainer(train_p, d, device)
            trainer.load(model_path)
            train_size = None
            results[d] = train_size
            trained_models[d] = trainer
            continue

        print(f"\n→ No saved model found. Training new model...")
        chunk_start = datetime.now()
        mwpm_accuracy = None
        train_size = None

        try:
            trainer, train_size = train_until_beat_mwpm(
                p=train_p, d=d, device=device,
                max_train_size=max_train_size, chunk_size=chunk_size
            )

            # Get final accuracies for logging
            if trainer is not None:
                print("\nFinal evaluation...")
                gnn_accuracy = trainer.evaluate(num_samples=10000)
                mwpm_accuracy = get_mwpm_accuracy(train_p, d, num_shots=10000)
                training_time = (datetime.now() - chunk_start).total_seconds() / 60

                trainer.save(model_path)
                print(f"✓ Model saved to {model_path}")

                # Log successful completion
                log_training_result(
                    d=d,
                    p=train_p,
                    train_size=train_size,
                    gnn_accuracy=gnn_accuracy,
                    mwpm_accuracy=mwpm_accuracy,
                    training_time=training_time,
                    status="completed",
                    notes=f"5 feat, 4 layer, h=128"
                )
            else:
                print(f"Training failed for d={d}")
                continue

        except KeyboardInterrupt:
            print(f"\n⚠ Training interrupted for d={d}")
            training_time = (datetime.now() - chunk_start).total_seconds() / 60

            # Log partial result (without gnn_accuracy)
            log_training_result(
                d=d,
                p=train_p,
                train_size=train_size if train_size else "partial",
                gnn_accuracy=None,  # Mark as incomplete
                mwpm_accuracy=mwpm_accuracy,
                training_time=training_time,
                status="interrupted",
                notes=f"5 feat, 4 layer, h=128"
            )
            raise  # Re-raise to stop loop

        trained_models[d] = trainer
        results[d] = train_size

except KeyboardInterrupt:
    print(f"\n\n{'═'*60}")
    print("TRAINING INTERRUPTED")
    print(f"{'═'*60}")
    print("✓ All completed distances logged to TRAINING_LOG.md")
    print(f"{'═'*60}\n")

# Print summary
print(f"\n{'═'*60}")
print("SUMMARY")
print(f"{'═'*60}")
for d in [3, 5, 7]:
    if d in results:
        if results[d]:
            print(f"  d={d}: ✓ Completed ({results[d]:,} samples)")
        else:
            print(f"  d={d}: ℹ Loaded from weights/")
    else:
        print(f"  d={d}: ✗ Not trained")
print(f"{'═'*60}")


████████████████████████████████████████████████████████████
█  DISTANCE 3
████████████████████████████████████████████████████████████

→ No saved model found. Training new model...

############################################################
# PROGRESSIVE TRAINING: d=3, p=0.005
############################################################

STEP 1: Getting MWPM baseline
Computing MWPM baseline (100,000 shots)...
✓ MWPM Accuracy: 0.983450 (LER: 0.016550)

🎯 Target to beat: 0.983450

STEP 2: Setting up GNN

Initializing GNN Trainer for d=3, p=0.005

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors, 48 connections
  Spatial edges: 32
  Temporal edges: 16
  Node features: 5 (syndrome + 4 static)
  Average neighbors per detector: 2.0

[3/3] Building neural network...
✓ Model built: 62,209 parameters

Ready to train!


STEP 3: Progressive training

────────────────────────────────────────────────────────────
📊 Training with 100 sample

    Epoch 1/10: Loss=1.2569, Train Acc=0.1000


    Epoch 2/10: Loss=1.2372, Train Acc=0.1000


    Epoch 3/10: Loss=1.2203, Train Acc=0.5400


    Epoch 4/10: Loss=1.2052, Train Acc=0.5700


...

    Epoch 1/1: Loss=0.6195, Train Acc=0.8335

  Evaluating on fresh test data...

  📈 Results:
     GNN Accuracy:  0.861500
     MWPM Accuracy: 0.985850
     Difference:    -12.435%

  ❌ Not yet. Trying 10x more data...

────────────────────────────────────────────────────────────
📊 Training with 100,000,000 samples
────────────────────────────────────────────────────────────

  Chunk 1/10: 10,000,000 samples
  Generating data...


    Epoch 1/1: Loss=0.6196, Train Acc=0.8366

  Chunk 2/10: 10,000,000 samples
  Generating data...


    Epoch 1/1: Loss=0.5625, Train Acc=0.8653

  Chunk 3/10: 10,000,000 samples
  Generating data...


    Epoch 1/1: Loss=0.5401, Train Acc=0.8774

  Chunk 4/10: 10,000,000 samples
  Generating data...


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# QUICK TEST - Run this first to verify everything works
# ============================================================================

print("🧪 Quick Test: Training a small GNN on d=3")
print("="*50)

# Create trainer
test_trainer = GNNTrainer(p=0.005, d=3, device=device)

# Generate small dataset
print("\nGenerating 10,000 training samples...")
syndromes, labels = test_trainer.sample_data(10000)
print(f"✓ Syndromes shape: {syndromes.shape}")
print(f"✓ Labels shape: {labels.shape}")
print(f"✓ Example syndrome: {syndromes[0, :5].tolist()} ...")

# Train for 3 epochs
print("\nTraining for 3 epochs...")
for epoch in range(3):
    loss, acc = test_trainer.train_epoch(syndromes, labels, batch_size=256)
    print(f"  Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={acc:.4f}")

# Evaluate
print("\nEvaluating on fresh test data...")
test_acc = test_trainer.evaluate(5000)
print(f"✓ Test Accuracy: {test_acc:.4f}")

print("\n" + "="*50)
print("✅ Quick test passed! The GNN is working.")
print("="*50)

🧪 Quick Test: Training a small GNN on d=3

Initializing GNN Trainer for d=3, p=0.005

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors, 48 connections
  Average neighbors per detector: 2.0

[3/3] Building neural network...
✓ Model built: 59,137 parameters

Ready to train!


Generating 10,000 training samples...
✓ Syndromes shape: torch.Size([10000, 24])
✓ Labels shape: torch.Size([10000])
✓ Example syndrome: [-1, -1, -1, -1, -1] ...

Training for 3 epochs...


  Epoch 1: Loss=0.4129, Accuracy=0.8907


  Epoch 2: Loss=0.2895, Accuracy=0.8960


  Epoch 3: Loss=0.2861, Accuracy=0.8960

Evaluating on fresh test data...
✓ Test Accuracy: 0.9024

✅ Quick test passed! The GNN is working.


# Sparse GNN Implementation (v2)

## Key Changes from Dense Implementation:
1. **Sparse graphs** - Only triggered detectors become nodes
2. **DEM edge weights** - Uses detector error model probabilities  
3. **Zero-tensor for empty graphs** - No virtual nodes needed
4. **Two output heads** - Predicts both λ_Z and λ_X
5. **Residual connections** - Preserves node features through layers
6. **Continuous edge weights** - Linear projection instead of embedding

In [ ]:
# ============================================================================
# SPARSE GNN V2: Core Components
# ============================================================================

class DEMGraph:
    """
    Builds sparse graph structure from Detector Error Model.
    Pre-computes which detector pairs can be connected and their probabilities.
    """

    def __init__(self, circuit, device):
        self.device = device
        self.circuit = circuit
        self.num_detectors = circuit.num_detectors

        # Get detector coordinates for node features
        self.detector_coords = circuit.get_detector_coordinates()

        # Build DEM edge lookup: which detectors can be connected
        self.dem_edges, self.dem_probs = self._build_dem_lookup()

        # Compute static node features (will be indexed by triggered detectors)
        self.static_features = self._build_static_features()

        print(f"✓ DEMGraph built: {self.num_detectors} detectors")
        print(f"  Possible edge pairs: {len(self.dem_edges)}")
        print(f"  Static features per node: {self.static_features.shape[1]}")

    def _build_dem_lookup(self):
        """
        Extract detector-detector connections from the Detector Error Model.
        Returns dict: (det_i, det_j) -> probability
        """
        dem = self.circuit.detector_error_model(decompose_errors=True)

        edges = {}  # (i, j) -> probability

        for instruction in dem:
            if instruction.type == "error":
                prob = instruction.args_copy()[0]
                targets = instruction.targets_copy()

                # Get detector IDs from this error
                detectors = []
                for t in targets:
                    if t.is_relative_detector_id():
                        detectors.append(t.val)

                # Connect all pairs of detectors triggered by same error
                for i in range(len(detectors)):
                    for j in range(i + 1, len(detectors)):
                        d1, d2 = detectors[i], detectors[j]
                        key = (min(d1, d2), max(d1, d2))
                        # Accumulate probabilities (errors can overlap)
                        edges[key] = edges.get(key, 0) + prob

        return edges, edges  # Same dict, just for clarity

    def _build_static_features(self):
        """
        Build static features for all detectors.
        Features: [stabilizer_type, x_norm, y_norm, t_norm]
        """
        coords = self.detector_coords

        # Get ranges for normalization
        all_coords = list(coords.values())
        if len(all_coords) == 0:
            return torch.zeros((self.num_detectors, 4), dtype=torch.float)

        xs = [c[0] for c in all_coords]
        ys = [c[1] for c in all_coords]
        ts = [c[2] for c in all_coords]

        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        t_max = max(ts)

        features = []
        for det_id in range(self.num_detectors):
            coord = coords.get(det_id, [0, 0, 0])
            x, y, t = coord[0], coord[1], coord[2]

            # Stabilizer type from checkerboard pattern
            stab_type = ((x + y) / 2) % 2

            # Normalized positions
            x_norm = (x - x_min) / max(1, x_max - x_min)
            y_norm = (y - y_min) / max(1, y_max - y_min)
            t_norm = t / max(1, t_max)

            features.append([stab_type, x_norm, y_norm, t_norm])

        return torch.tensor(features, dtype=torch.float)

    def create_sparse_batch(self, detections):
        """
        Create batched sparse graph from detection events.

        Args:
            detections: [batch_size, num_detectors] boolean tensor

        Returns:
            Data object with sparse graph, or None if all empty
        """
        batch_size = detections.shape[0]

        all_x = []           # Node features
        all_edges = []       # Edge indices
        all_weights = []     # Edge weights (DEM probabilities)
        all_batch = []       # Batch assignment

        node_offset = 0

        for b in range(batch_size):
            # Find triggered detectors in this sample
            triggered = detections[b].nonzero(as_tuple=True)[0]
            num_triggered = len(triggered)

            if num_triggered == 0:
                # Empty graph for this sample - will be handled by zero-tensor
                continue

            # Node features for triggered detectors
            triggered_np = triggered.cpu().numpy()
            features = self.static_features[triggered_np]  # [num_triggered, 4]
            all_x.append(features)

            # Build edges between triggered detectors using DEM
            triggered_set = set(triggered_np.tolist())
            local_id = {det: i for i, det in enumerate(triggered_np)}

            for (d1, d2), prob in self.dem_edges.items():
                if d1 in triggered_set and d2 in triggered_set:
                    # Add bidirectional edge
                    i, j = local_id[d1], local_id[d2]
                    all_edges.append([node_offset + i, node_offset + j])
                    all_edges.append([node_offset + j, node_offset + i])
                    # Convert probability to weight: -log(p)
                    weight = -np.log(max(prob, 1e-10))
                    all_weights.extend([weight, weight])

            # Batch assignment
            all_batch.extend([b] * num_triggered)
            node_offset += num_triggered

        # Handle case where ALL samples have zero triggers
        if len(all_x) == 0:
            return None

        # Stack tensors
        x = torch.cat(all_x, dim=0).to(self.device)
        batch = torch.tensor(all_batch, dtype=torch.long, device=self.device)

        if len(all_edges) > 0:
            edge_index = torch.tensor(all_edges, dtype=torch.long, device=self.device).t().contiguous()
            edge_weight = torch.tensor(all_weights, dtype=torch.float, device=self.device)
        else:
            # No edges (isolated nodes)
            edge_index = torch.zeros((2, 0), dtype=torch.long, device=self.device)
            edge_weight = torch.zeros(0, dtype=torch.float, device=self.device)

        return Data(
            x=x,
            edge_index=edge_index,
            edge_weight=edge_weight,
            batch=batch,
            num_graphs=batch_size
        )

In [ ]:
# ============================================================================
# SPARSE GNN V2: Attention Layer with Continuous Edge Weights
# ============================================================================

class GATConvContinuous(MessagePassing):
    """
    Graph Attention Layer with continuous edge weights (from DEM).

    Instead of embedding discrete edge types, projects continuous
    edge weight (−log probability) into the attention space.
    """

    def __init__(self, in_channels, out_channels, heads=4, concat=True, dropout=0.0):
        super().__init__(aggr='add', node_dim=0)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        # Node feature transformation
        self.lin = torch.nn.Linear(in_channels, heads * out_channels, bias=False)

        # Edge weight projection: scalar -> head_dim per head
        self.edge_proj = torch.nn.Linear(1, heads * out_channels)

        # Attention: a^T [source || target || edge_proj]
        self.att = torch.nn.Parameter(torch.Tensor(1, heads, 3 * out_channels))

        # Output bias
        if concat:
            self.bias = torch.nn.Parameter(torch.Tensor(heads * out_channels))
        else:
            self.bias = torch.nn.Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.lin.weight)
        torch.nn.init.xavier_uniform_(self.edge_proj.weight)
        torch.nn.init.zeros_(self.edge_proj.bias)
        torch.nn.init.xavier_uniform_(self.att)
        torch.nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_weight):
        """
        Args:
            x: Node features [num_nodes, in_channels]
            edge_index: [2, num_edges]
            edge_weight: [num_edges] continuous weights
        """
        # Handle empty edge case
        if edge_index.shape[1] == 0:
            # No edges - just transform node features
            x = self.lin(x).view(-1, self.heads, self.out_channels)
            if self.concat:
                return x.view(-1, self.heads * self.out_channels) + self.bias
            else:
                return x.mean(dim=1) + self.bias

        # Transform node features
        x = self.lin(x).view(-1, self.heads, self.out_channels)

        # Project edge weights: [num_edges] -> [num_edges, heads, out_channels]
        edge_emb = self.edge_proj(edge_weight.unsqueeze(-1))
        edge_emb = edge_emb.view(-1, self.heads, self.out_channels)

        # Message passing
        out = self.propagate(edge_index, x=x, edge_emb=edge_emb)

        if self.concat:
            out = out.view(-1, self.heads * self.out_channels)
        else:
            out = out.mean(dim=1)

        return out + self.bias

    def message(self, x_i, x_j, edge_emb, index, ptr, size_i):
        """Compute attention and messages."""
        # Concatenate: [source || target || edge_weight_projection]
        alpha_input = torch.cat([x_i, x_j, edge_emb], dim=-1)

        # Attention scores
        alpha = (alpha_input * self.att).sum(dim=-1)
        alpha = F.leaky_relu(alpha, negative_slope=0.2)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        return x_j * alpha.unsqueeze(-1)

In [ ]:
# ============================================================================
# SPARSE GNN V2: Decoder with Two Heads and Residual Connections
# ============================================================================

class SparseDecoder(torch.nn.Module):
    """
    Sparse GNN decoder with:
    - Continuous edge weights (DEM probabilities)
    - Residual connections
    - Two output heads (λ_Z and λ_X)
    """

    def __init__(self, hidden_dim=128, num_layers=4, num_heads=4, num_input_features=4):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Input projection (static features -> hidden)
        self.input_proj = torch.nn.Linear(num_input_features, hidden_dim)

        # GNN layers with attention
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        for _ in range(num_layers):
            self.layers.append(GATConvContinuous(
                hidden_dim,
                hidden_dim // num_heads,
                heads=num_heads,
                concat=True,
                dropout=0.1
            ))
            self.norms.append(torch.nn.LayerNorm(hidden_dim))

        # Two output heads
        self.head_Z = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 64),
            torch.nn.SiLU(),
            torch.nn.Linear(64, 1)
        )

        self.head_X = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 64),
            torch.nn.SiLU(),
            torch.nn.Linear(64, 1)
        )

        # Learned embedding for empty graphs (no triggers)
        self.empty_graph_embedding = torch.nn.Parameter(torch.zeros(hidden_dim))

    def forward(self, data):
        """
        Forward pass.

        Returns:
            (logits_Z, logits_X): Both [batch_size, 1]
        """
        # Handle empty graph case
        if data is None:
            # All samples have zero triggers - return zeros
            # Caller must provide batch_size separately
            raise ValueError("Empty data - use forward_with_empty() instead")

        x = data.x
        edge_index = data.edge_index
        edge_weight = data.edge_weight
        batch = data.batch
        num_graphs = data.num_graphs

        # Input projection
        x = self.input_proj(x)

        # GNN layers with residual connections
        for layer, norm in zip(self.layers, self.norms):
            x_new = layer(x, edge_index, edge_weight)
            x_new = norm(x_new)
            x = x + F.silu(x_new)  # Residual connection

        # Pool nodes to graph embeddings
        graph_emb = global_mean_pool(x, batch)  # [num_graphs_with_nodes, hidden]

        # Handle graphs that had zero triggers (they're missing from batch)
        # We need to insert zero embeddings for them
        full_emb = self._expand_embeddings(graph_emb, batch, num_graphs)

        # Two-head predictions
        logits_Z = self.head_Z(full_emb)
        logits_X = self.head_X(full_emb)

        return logits_Z, logits_X

    def _expand_embeddings(self, graph_emb, batch, num_graphs):
        """
        Expand embeddings to include empty graphs.

        Some graphs may have had 0 triggered detectors and thus
        don't appear in the batch. Fill those with the learned
        empty_graph_embedding.
        """
        # Find which graph indices actually have nodes
        unique_graphs = batch.unique()

        if len(unique_graphs) == num_graphs:
            # All graphs have nodes, no expansion needed
            return graph_emb

        # Create full embedding tensor
        device = graph_emb.device
        full_emb = self.empty_graph_embedding.unsqueeze(0).expand(num_graphs, -1).clone()

        # Fill in graphs that have embeddings
        for idx, graph_idx in enumerate(unique_graphs):
            full_emb[graph_idx] = graph_emb[idx]

        return full_emb

In [ ]:
# ============================================================================
# SPARSE GNN V2: Trainer with Two-Head Loss
# ============================================================================

class SparseGNNTrainer:
    """
    Trainer for Sparse GNN decoder.

    Key differences from dense trainer:
    - Uses DEMGraph for sparse graph construction
    - Two-head output with masked loss
    - Supports both memory-Z and memory-X experiments
    """

    def __init__(self, p, d, device, hidden_dim=128, num_layers=4):
        self.p = p
        self.d = d
        self.device = device

        print(f"\n{'='*60}")
        print(f"Initializing SPARSE GNN Trainer for d={d}, p={p}")
        print(f"{'='*60}")

        # Build circuit
        print("\n[1/4] Building quantum circuit...")
        self.circuit = surface_code_circuit(p, d)

        # Build DEM-based sparse graph structure
        print("\n[2/4] Building DEM graph structure...")
        self.dem_graph = DEMGraph(self.circuit, device)

        # Build model (4 input features: stab_type, x, y, t)
        print("\n[3/4] Building sparse neural network...")
        self.model = SparseDecoder(
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_input_features=4
        ).to(device)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=3e-4)

        # Count parameters
        num_params = sum(p.numel() for p in self.model.parameters())
        print(f"✓ Model built: {num_params:,} parameters")

        print("\n[4/4] Ready!")
        print(f"{'='*60}\n")

    def sample_data(self, num_samples, experiment='Z'):
        """
        Generate syndrome samples.

        Args:
            num_samples: Number of samples
            experiment: 'Z' for memory-Z, 'X' for memory-X, 'both' for alternating

        Returns:
            detections: [batch, num_detectors] boolean
            labels: [batch, 2] where [:, 0] is λ_Z, [:, 1] is λ_X
                    NaN indicates missing label
        """
        sampler = self.circuit.compile_detector_sampler()
        detections, flips = sampler.sample(num_samples, separate_observables=True)

        detections = torch.from_numpy(detections.astype(np.float32)).to(self.device)

        # Labels: [batch, 2] - for now we only have Z labels from memory-Z
        labels = torch.full((num_samples, 2), float('nan'), device=self.device)
        labels[:, 0] = torch.from_numpy(flips.astype(np.float32).flatten()).to(self.device)

        return detections, labels

    def train_epoch(self, detections, labels, batch_size=256):
        """Train one epoch with masked two-head loss."""
        self.model.train()
        num_samples = detections.shape[0]
        num_batches = math.ceil(num_samples / batch_size)

        total_loss = 0
        correct_Z = 0
        total_Z = 0

        with trange(num_batches, desc="Training", leave=False) as pbar:
            for batch_idx in pbar:
                start = batch_idx * batch_size
                end = min(start + batch_size, num_samples)

                batch_det = detections[start:end]
                batch_lab = labels[start:end]
                actual_batch_size = end - start

                # Create sparse graph batch
                batch_data = self.dem_graph.create_sparse_batch(batch_det)

                if batch_data is not None:
                    batch_data.num_graphs = actual_batch_size
                    logits_Z, logits_X = self.model(batch_data)
                else:
                    # All empty graphs - use empty embedding
                    empty_emb = self.model.empty_graph_embedding.unsqueeze(0)
                    empty_emb = empty_emb.expand(actual_batch_size, -1)
                    logits_Z = self.model.head_Z(empty_emb)
                    logits_X = self.model.head_X(empty_emb)

                # Masked loss - only compute for non-NaN labels
                loss = 0

                # Z head loss
                mask_Z = ~torch.isnan(batch_lab[:, 0])
                if mask_Z.sum() > 0:
                    loss_Z = F.binary_cross_entropy_with_logits(
                        logits_Z[mask_Z].squeeze(),
                        batch_lab[mask_Z, 0]
                    )
                    loss = loss + loss_Z

                    # Track accuracy
                    pred_Z = (torch.sigmoid(logits_Z[mask_Z]) > 0.5).float().squeeze()
                    correct_Z += (pred_Z == batch_lab[mask_Z, 0]).sum().item()
                    total_Z += mask_Z.sum().item()

                # X head loss (currently all NaN, but ready for future)
                mask_X = ~torch.isnan(batch_lab[:, 1])
                if mask_X.sum() > 0:
                    loss_X = F.binary_cross_entropy_with_logits(
                        logits_X[mask_X].squeeze(),
                        batch_lab[mask_X, 1]
                    )
                    loss = loss + loss_X

                # Backward
                self.optimizer.zero_grad()
                if isinstance(loss, torch.Tensor):
                    loss.backward()
                    self.optimizer.step()
                    total_loss += loss.item()

                # Progress bar
                acc_Z = correct_Z / total_Z if total_Z > 0 else 0
                pbar.set_postfix({'loss': f'{loss.item() if isinstance(loss, torch.Tensor) else 0:.4f}',
                                  'acc_Z': f'{acc_Z:.4f}'})

        avg_loss = total_loss / num_batches
        accuracy_Z = correct_Z / total_Z if total_Z > 0 else 0
        return avg_loss, accuracy_Z

    def evaluate(self, num_samples=10000, batch_size=500):
        """Evaluate on fresh data."""
        self.model.eval()
        detections, labels = self.sample_data(num_samples)

        correct_Z = 0
        total_Z = 0

        with torch.no_grad():
            for i in range(0, num_samples, batch_size):
                batch_det = detections[i:i+batch_size]
                batch_lab = labels[i:i+batch_size]
                actual_batch_size = batch_det.shape[0]

                batch_data = self.dem_graph.create_sparse_batch(batch_det)

                if batch_data is not None:
                    batch_data.num_graphs = actual_batch_size
                    logits_Z, logits_X = self.model(batch_data)
                else:
                    empty_emb = self.model.empty_graph_embedding.unsqueeze(0)
                    empty_emb = empty_emb.expand(actual_batch_size, -1)
                    logits_Z = self.model.head_Z(empty_emb)

                mask_Z = ~torch.isnan(batch_lab[:, 0])
                if mask_Z.sum() > 0:
                    pred_Z = (torch.sigmoid(logits_Z[mask_Z]) > 0.5).float().squeeze()
                    correct_Z += (pred_Z == batch_lab[mask_Z, 0]).sum().item()
                    total_Z += mask_Z.sum().item()

        return correct_Z / total_Z if total_Z > 0 else 0

    def get_ler(self, num_shots=100000, batch_size=1000):
        """Compute logical error rate."""
        accuracy = self.evaluate(num_shots, batch_size)
        return 1 - accuracy

    def save(self, path):
        """Save model weights."""
        torch.save(self.model.state_dict(), path)
        print(f"✓ Model saved to {path}")

    def load(self, path):
        """Load model weights."""
        self.model.load_state_dict(torch.load(path, weights_only=True))
        print(f"✓ Model loaded from {path}")

In [ ]:
# ============================================================================
# SPARSE GNN V2: Progressive Training (Same Strategy as Dense)
# ============================================================================

def train_sparse_until_beat_mwpm(p, d, device, max_train_size=10**8, chunk_size=10**7):
    """Train sparse GNN with progressively larger datasets until beating MWPM."""
    import gc

    print(f"\n{'#'*60}")
    print(f"# SPARSE GNN PROGRESSIVE TRAINING: d={d}, p={p}")
    print(f"{'#'*60}")

    # Get MWPM baseline
    print("\n" + "="*60)
    print("STEP 1: Getting MWPM baseline")
    print("="*60)
    mwpm_accuracy = get_mwpm_accuracy(p, d)
    print(f"\n🎯 Target to beat: {mwpm_accuracy:.6f}")

    # Initialize sparse trainer
    print("\n" + "="*60)
    print("STEP 2: Setting up Sparse GNN")
    print("="*60)
    trainer = SparseGNNTrainer(p, d, device)

    # Progressive training
    print("\n" + "="*60)
    print("STEP 3: Progressive training")
    print("="*60)

    train_size = 1000  # Start larger for sparse (more efficient)
    beat_mwpm = False

    while train_size <= max_train_size and not beat_mwpm:
        print(f"\n{'─'*60}")
        print(f"📊 Training with {train_size:,} samples")
        print(f"{'─'*60}")

        # Reset model
        trainer.model = SparseDecoder(hidden_dim=128, num_layers=4, num_input_features=4).to(device)
        trainer.optimizer = torch.optim.Adam(trainer.model.parameters(), lr=3e-4)

        # Calculate chunks
        num_chunks = max(1, train_size // chunk_size)
        samples_per_chunk = min(train_size, chunk_size)

        # Train in chunks
        for chunk_idx in range(num_chunks):
            current_chunk_size = min(samples_per_chunk, train_size - chunk_idx * chunk_size)

            print(f"\n  Chunk {chunk_idx + 1}/{num_chunks}: {current_chunk_size:,} samples")

            detections, labels = trainer.sample_data(current_chunk_size)

            # Multiple epochs for small datasets
            num_epochs = max(1, min(10, 10000 // (current_chunk_size // 256 + 1)))

            for epoch in range(num_epochs):
                loss, acc = trainer.train_epoch(detections, labels, batch_size=256)
                print(f"    Epoch {epoch+1}/{num_epochs}: Loss={loss:.4f}, Acc_Z={acc:.4f}")

            del detections, labels
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Evaluate
        print(f"\n  Evaluating on fresh test data...")
        gnn_accuracy = trainer.evaluate(num_samples=10000)

        print(f"\n  📈 Results:")
        print(f"     Sparse GNN Accuracy: {gnn_accuracy:.6f}")
        print(f"     MWPM Accuracy:       {mwpm_accuracy:.6f}")
        print(f"     Difference:          {(gnn_accuracy - mwpm_accuracy)*100:+.3f}%")

        if gnn_accuracy > mwpm_accuracy:
            print(f"\n  🎉 SUCCESS! Sparse GNN beats MWPM at train_size = {train_size:,}")
            beat_mwpm = True
        else:
            print(f"\n  ❌ Not yet. Trying 10x more data...")
            train_size *= 10
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not beat_mwpm:
        print(f"\n❌ Did not beat MWPM within max train_size = {max_train_size:,}")
        return None, None

    return trainer, train_size

In [ ]:
# ============================================================================
# SPARSE GNN V2: Main Training Loop with Logging
# ============================================================================

def get_sparse_model_name(d, p, layers=4, hidden_dim=128):
    """Generate descriptive model filename for sparse version."""
    return f"sparse_gnn_d{d}_p{p}_layer{layers}_hidden{hidden_dim}.pt"

def run_sparse_training():
    """Run full sparse GNN training with logging."""

    # Configuration
    train_p = 0.005
    max_train_size = 10**8
    chunk_size = 10**7

    results = {}
    trained_models = {}
    start_time = datetime.now()

    # Create weights directory
    weights_dir = "weights"
    if not os.path.exists(weights_dir):
        os.makedirs(weights_dir)
        print(f"✓ Created {weights_dir}/ directory")

    try:
        for d in [3, 5, 7]:
            model_path = os.path.join(weights_dir, get_sparse_model_name(d, train_p))

            print(f"\n{'█'*60}")
            print(f"█  SPARSE GNN - DISTANCE {d}")
            print(f"{'█'*60}")

            # Check if model exists
            if os.path.exists(model_path):
                print(f"\n✓ Found existing model: {model_path}")
                trainer = SparseGNNTrainer(train_p, d, device)
                trainer.load(model_path)
                results[d] = None
                trained_models[d] = trainer
                continue

            print(f"\n→ Training new sparse model...")
            chunk_start = datetime.now()
            mwpm_accuracy = None
            train_size = None

            try:
                trainer, train_size = train_sparse_until_beat_mwpm(
                    p=train_p, d=d, device=device,
                    max_train_size=max_train_size, chunk_size=chunk_size
                )

                if trainer is not None:
                    print("\nFinal evaluation...")
                    gnn_accuracy = trainer.evaluate(num_samples=10000)
                    mwpm_accuracy = get_mwpm_accuracy(train_p, d, num_shots=10000)
                    training_time = (datetime.now() - chunk_start).total_seconds() / 60

                    trainer.save(model_path)

                    # Log to training log
                    log_training_result(
                        d=d,
                        p=train_p,
                        train_size=train_size,
                        gnn_accuracy=gnn_accuracy,
                        mwpm_accuracy=mwpm_accuracy,
                        training_time=training_time,
                        status="completed",
                        notes="SPARSE v2: DEM weights, 2-head, residual"
                    )
                else:
                    print(f"Training failed for d={d}")
                    continue

            except KeyboardInterrupt:
                print(f"\n⚠ Training interrupted for d={d}")
                training_time = (datetime.now() - chunk_start).total_seconds() / 60

                log_training_result(
                    d=d,
                    p=train_p,
                    train_size=train_size if train_size else "partial",
                    gnn_accuracy=None,
                    mwpm_accuracy=mwpm_accuracy,
                    training_time=training_time,
                    status="interrupted",
                    notes="SPARSE v2: DEM weights, 2-head, residual"
                )
                raise

            trained_models[d] = trainer
            results[d] = train_size

    except KeyboardInterrupt:
        print(f"\n\n{'═'*60}")
        print("TRAINING INTERRUPTED")
        print(f"{'═'*60}")

    # Summary
    print(f"\n{'═'*60}")
    print("SPARSE GNN SUMMARY")
    print(f"{'═'*60}")
    for d in [3, 5, 7]:
        if d in results:
            if results[d]:
                print(f"  d={d}: ✓ Completed ({results[d]:,} samples)")
            else:
                print(f"  d={d}: ℹ Loaded from weights/")
        else:
            print(f"  d={d}: ✗ Not trained")
    print(f"{'═'*60}")

    return trained_models, results

In [ ]:
# ============================================================================
# SPARSE GNN V2: Quick Test
# ============================================================================

print("🧪 Quick Test: Sparse GNN on d=3")
print("="*50)

# Create sparse trainer
sparse_trainer = SparseGNNTrainer(p=0.005, d=3, device=device)

# Generate small dataset
print("\nGenerating 10,000 training samples...")
detections, labels = sparse_trainer.sample_data(10000)
print(f"✓ Detections shape: {detections.shape}")
print(f"✓ Labels shape: {labels.shape}")

# Check sparsity
num_triggered = (detections > 0).sum(dim=1).float()
print(f"✓ Avg triggered detectors per sample: {num_triggered.mean():.2f}")
print(f"  (vs {sparse_trainer.dem_graph.num_detectors} total detectors)")

# Train for 3 epochs
print("\nTraining for 3 epochs...")
for epoch in range(3):
    loss, acc = sparse_trainer.train_epoch(detections, labels, batch_size=256)
    print(f"  Epoch {epoch+1}: Loss={loss:.4f}, Accuracy_Z={acc:.4f}")

# Evaluate
print("\nEvaluating on fresh test data...")
test_acc = sparse_trainer.evaluate(5000)
print(f"✓ Test Accuracy: {test_acc:.4f}")

print("\n" + "="*50)
print("✅ Sparse GNN quick test passed!")
print("="*50)

In [ ]:
# ============================================================================
# RUN SPARSE GNN TRAINING
# ============================================================================
# Uncomment below to run full training:

# trained_models, results = run_sparse_training()